In [ ]:
# %pip install duckdb
# %pip install pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:

import duckdb
import pandas as pd

In [4]:
# Create a conection to the database
conn = duckdb.connect("../bren-eds213-data/database/database.duckdb")

2. Now let’s query the databse using:
.sql() method – specific to DuckDB and analysis focused

In [ ]:
# First method
conn.sql("SELECT * FROM Site LIMIT 5")

┌─────────┬─────────────────────┬─────────────────┬──────────┬───────────┬───────┐
│  Code   │      Site_name      │    Location     │ Latitude │ Longitude │ Area  │
│ varchar │       varchar       │     varchar     │  float   │   float   │ float │
├─────────┼─────────────────────┼─────────────────┼──────────┼───────────┼───────┤
│ barr    │ Barrow              │ Alaska, USA     │     71.3 │    -156.6 │ 220.4 │
│ burn    │ Burntpoint Creek    │ Ontario, Canada │     55.2 │     -84.3 │  63.0 │
│ bylo    │ Bylot Island        │ Nunavut, Canada │     73.2 │     -80.0 │ 723.6 │
│ cakr    │ Cape Krusenstern    │ Alaska, USA     │     67.1 │    -163.5 │  54.1 │
│ cari    │ Canning River Delta │ Alaska, USA     │     70.1 │    -145.8 │ 722.0 │
└─────────┴─────────────────────┴─────────────────┴──────────┴───────────┴───────┘

Note: This is also a lazy evaluation like we were doing with dbplyr. At this point, the data has not been fully processed or brought into Python memory yet. It’s a symbolic representation of a query. To see the actual data, you need to materialize the relation. You can do this in several ways:

.show(): Prints a nice tabular representation (great for interactive exploration). .fetchall(): Returns all results as a list of tuples (the traditional DB-API way). .fetchone(): Returns the next single row as a tuple. .df(): Converts the result into a Pandas DataFrame.Now we want results…

In [ ]:
# Add the fetch get the data
# You get a list of tuples (one per row)
conn.sql("SELECT * FROM Site LIMIT 5"). fetchall()

[('barr',
  'Barrow',
  'Alaska, USA',
  71.30000305175781,
  -156.60000610351562,
  220.39999389648438),
 ('burn',
  'Burntpoint Creek',
  'Ontario, Canada',
  55.20000076293945,
  -84.30000305175781,
  63.0),
 ('bylo',
  'Bylot Island',
  'Nunavut, Canada',
  73.19999694824219,
  -80.0,
  723.5999755859375),
 ('cakr',
  'Cape Krusenstern',
  'Alaska, USA',
  67.0999984741211,
  -163.5,
  54.099998474121094),
 ('cari',
  'Canning River Delta',
  'Alaska, USA',
  70.0999984741211,
  -145.8000030517578,
  722.0)]

In [7]:
# You get a pandas dataframe
site_df = conn.sql("SELECT * FROM Site").df()
site_df.head()

,Code,Site_name,Location,Latitude,Longitude,Area
0,barr,Barrow,"Alaska, USA",71.300003,-156.600006,220.399994
1,burn,Burntpoint Creek,"Ontario, Canada",55.200001,-84.300003,63.000000
2,bylo,Bylot Island,"Nunavut, Canada",73.199997,-80.000000,723.599976
3,cakr,Cape Krusenstern,"Alaska, USA",67.099998,-163.500000,54.099998
4,cari,Canning River Delta,"Alaska, USA",70.099998,-145.800003,722.000000


In [8]:
site_table = conn.execute("SELECT * FROM Site")

site_table

## Using a cursor
- Go row by row
b. .execute() method – more ubiquitous to other database workflows where you create a cursor object that you use to iterate on the rows of a table

In [9]:
cur = conn.cursor()

In [10]:
cur.execute("SELECT * FROM Site LIMIT 5")

In [ ]:
# you can apply the fetchall()...
cur.fetchall()

[('barr',
  'Barrow',
  'Alaska, USA',
  71.30000305175781,
  -156.60000610351562,
  220.39999389648438),
 ('burn',
  'Burntpoint Creek',
  'Ontario, Canada',
  55.20000076293945,
  -84.30000305175781,
  63.0),
 ('bylo',
  'Bylot Island',
  'Nunavut, Canada',
  73.19999694824219,
  -80.0,
  723.5999755859375),
 ('cakr',
  'Cape Krusenstern',
  'Alaska, USA',
  67.0999984741211,
  -163.5,
  54.099998474121094),
 ('cari',
  'Canning River Delta',
  'Alaska, USA',
  70.0999984741211,
  -145.8000030517578,
  722.0)]

In [12]:
# Always get tuples, even if you only request one column
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")

In [13]:
cur.fetchall()

[('14HPE1',),
 ('11eaba',),
 ('11eabaagc01',),
 ('11eabaagv01',),
 ('11eababbc02',),
 ('11eababsv01',),
 ('11eabaduh01',),
 ('11eabaduv01',),
 ('11eabarpc01',),
 ('11eabarpc02',)]

In [14]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")
[t[0] for t in cur.fetchall()]

['14HPE1',
 '11eaba',
 '11eabaagc01',
 '11eabaagv01',
 '11eababbc02',
 '11eababsv01',
 '11eabaduh01',
 '11eabaduv01',
 '11eabarpc01',
 '11eabarpc02']

Get the one result

In [21]:
# We want to iterate throught the rows and print the site code
cur.execute("SELECT * FROM Site")
for row in cur:
    print(f"Site code {row[0]}")

TypeError: '_duckdb.DuckDBPyConnection' object is not iterable

In [20]:
cur.execute("SELECT * FROM Site")

while True:
    row = cur.fetchone()
    if row == None:
        break
    print (f"Site code: {row[0]}")

Site code: barr
Site code: burn
Site code: bylo
Site code: cakr
Site code: cari
Site code: chau
Site code: chur
Site code: coat
Site code: colv
Site code: eaba
Site code: iglo
Site code: ikpi
Site code: lkri
Site code: made
Site code: nome
Site code: prba


In [24]:
# You can also create table 
cur.execute("""CREATE TEMP TABLE temp_nests AS
            SELECT * FROM Bird_nests""")

In [26]:
cur.execute("SELECT * FROM temp_nests LIMIT 5").fetchall()

[('b14.6',
  2014,
  'chur',
  '14HPE1',
  'sepl',
  'vloverti',
  datetime.date(2014, 6, 14),
  None,
  3,
  6.5,
  'float'),
 ('b11.7',
  2011,
  'eaba',
  '11eaba',
  'wrsa',
  'bhill',
  datetime.date(2011, 7, 10),
  'searcher',
  4,
  None,
  None),
 ('b11.6',
  2011,
  'eaba',
  '11eabaagc01',
  'amgp',
  'dkessler',
  datetime.date(2011, 6, 24),
  'searcher',
  4,
  6.0,
  'float'),
 ('b11.6',
  2011,
  'eaba',
  '11eabaagv01',
  'amgp',
  'dkessler',
  datetime.date(2011, 6, 25),
  'searcher',
  3,
  3.0,
  'float'),
 ('b11.6',
  2011,
  'eaba',
  '11eababbc02',
  'bbpl',
  'dkessler',
  datetime.date(2011, 6, 24),
  'searcher',
  4,
  4.0,
  'float')]

In [27]:
# Closing your connection
conn.close()